# 07. Boosting: Aprendiendo de los Errores

**Nivel:** 🔴 Avanzado  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [06. Random Forests](06-random-forests.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender la diferencia fundamental entre bagging y boosting
- Implementar AdaBoost desde cero y comprender su funcionamiento
- Comprender el marco teórico de Gradient Boosting
- Utilizar XGBoost y LightGBM para problemas reales
- Aplicar técnicas de regularización en boosting
- Tunear hiperparámetros para optimizar rendimiento

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
from sklearn.datasets import load_breast_cancer, make_classification, make_moons
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades
import sys
sys.path.append('../../shared/utils')
from visualization import plot_decision_boundary
from datasets import load_dataset

np.random.seed(42)
print("✅ Librerías importadas")

---
## 📌 1. Motivación: De la Colaboración a la Corrección

### Recordatorio: Random Forests (Bagging)

En Random Forests vimos:
```python
# Entrenar árboles en PARALELO (independientes)
for i in range(n_trees):
    bootstrap_sample = random_sample(data)
    tree[i].fit(bootstrap_sample)

# Votación democrática
prediction = majority_vote([tree[i].predict(x) for i in range(n_trees)])
```

**Ventaja:** Reduce varianza  
**Limitación:** No reduce bias (si árboles individuales son débiles, el ensemble también)

### Boosting: Un Enfoque Diferente

```python
# Entrenar árboles SECUENCIALMENTE (cada uno aprende de errores previos)
for i in range(n_trees):
    # Enfocarse en ejemplos que el modelo actual clasifica mal
    weights = calculate_sample_weights(current_errors)
    tree[i].fit(data, sample_weights=weights)
    update_ensemble(tree[i])

# Votación PONDERADA (mejores modelos tienen más voto)
prediction = weighted_vote([alpha[i] * tree[i].predict(x) for i in range(n_trees)])
```

**Ventaja:** Reduce bias Y varianza  
**Trade-off:** Más propenso a overfitting si no se regulariza

### Analogía: Estudio en Grupo

**Random Forests (Bagging):**
- 100 estudiantes estudian temas aleatorios independientemente
- En el examen, votan por la respuesta
- Cada estudiante es igualmente importante

**Boosting:**
- Primer estudiante estudia y hace el examen
- Segundo estudiante se enfoca en las preguntas que el primero falló
- Tercer estudiante se enfoca en lo que ambos anteriores fallaron
- ...
- Estudiantes más hábiles tienen más peso en la decisión final

### El Problema Real

**Detección de Fraude en Tarjetas de Crédito:**
- Dataset altamente desbalanceado (99.9% legítimo, 0.1% fraude)
- Costos asimétricos (fraude no detectado es muy costoso)
- Patrones complejos y cambiantes

**¿Por qué Boosting?**
- Se enfoca iterativamente en casos difíciles (fraudes)
- Puede construir fronteras de decisión muy complejas
- Maneja desbalance con sample weighting
- XGBoost/LightGBM son state-of-the-art en competencias Kaggle

### Aplicaciones Reales

- 🏆 **Kaggle**: Ganador de la mayoría de competencias estructuradas
- 🔍 **Web Search**: Ranking de resultados (ej: Bing usa LambdaMART)
- 💳 **Fintech**: Scoring crediticio, detección de fraude
- 🎯 **Advertising**: Predicción de click-through rate
- 🏥 **Healthcare**: Predicción de readmisiones hospitalarias

### La Pregunta Guía

> **¿Cómo puede un conjunto de modelos débiles (mejor que random guess) convertirse en un modelo arbitrariamente preciso?**

---
## 📊 2. Intuición Visual

In [ ]:
# Comparación: Random Forest vs AdaBoost
# Dataset con clases no linealmente separables
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar modelos
rf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
ada = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42)

rf.fit(X_train, y_train)
ada.fit(X_train, y_train)

# Crear grid
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predicciones
Z_rf = rf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
Z_ada = ada.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest
axes[0].contourf(xx, yy, Z_rf, alpha=0.3, cmap='RdBu')
axes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu', edgecolors='k')
axes[0].set_title(f'Random Forest\nTest Acc: {rf.score(X_test, y_test):.3f}', fontsize=14)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# AdaBoost
axes[1].contourf(xx, yy, Z_ada, alpha=0.3, cmap='RdBu')
axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu', edgecolors='k')
axes[1].set_title(f'AdaBoost\nTest Acc: {ada.score(X_test, y_test):.3f}', fontsize=14)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("\n💡 Observa:")
print("   • AdaBoost puede crear fronteras más complejas")
print("   • Ambos funcionan bien, pero con estrategias diferentes")

In [ ]:
# Visualizar el proceso iterativo de AdaBoost
# Entrenar AdaBoost paso a paso
n_estimators_list = [1, 2, 5, 10, 20, 50]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, n_est in enumerate(n_estimators_list):
    ada_temp = AdaBoostClassifier(n_estimators=n_est, learning_rate=1.0, random_state=42)
    ada_temp.fit(X_train, y_train)
    
    Z = ada_temp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    axes[idx].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu', 
                     edgecolors='k', s=20)
    axes[idx].set_title(f'n_estimators={n_est}\nAcc={ada_temp.score(X_test, y_test):.3f}',
                       fontsize=12)
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("\n💡 Observa cómo la frontera de decisión se vuelve más compleja con más estimadores")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Tres Algoritmos de Boosting

---

## 3.1 AdaBoost (Adaptive Boosting)

**Idea:** Aumentar el peso de ejemplos mal clasificados en cada iteración.

### Algoritmo

**Inicialización:**
$$
w_i^{(1)} = \frac{1}{n}, \quad i = 1, ..., n \tag{1}
$$

**Para t = 1 to T:**

1. Entrenar clasificador débil $h_t$ con pesos $w^{(t)}$

2. Calcular error ponderado:
$$
\epsilon_t = \frac{\sum_{i=1}^{n} w_i^{(t)} \mathbb{1}[y_i \neq h_t(x_i)]}{\sum_{i=1}^{n} w_i^{(t)}} \tag{2}
$$

3. Calcular peso del clasificador:
$$
\alpha_t = \frac{1}{2} \ln\left(\frac{1 - \epsilon_t}{\epsilon_t}\right) \tag{3}
$$

**Interpretación de $\alpha_t$:**
- Si $\epsilon_t = 0.5$ (random): $\alpha_t = 0$ (no contribuye)
- Si $\epsilon_t \to 0$ (perfecto): $\alpha_t \to \infty$ (mucho peso)
- Si $\epsilon_t > 0.5$ (peor que random): $\alpha_t < 0$ (invertir predicción)

4. Actualizar pesos de muestras:
$$
w_i^{(t+1)} = w_i^{(t)} \exp(-\alpha_t y_i h_t(x_i)) \tag{4}
$$

Normalizar: $w_i^{(t+1)} \leftarrow \frac{w_i^{(t+1)}}{\sum_j w_j^{(t+1)}}$

**Predicción final:**
$$
H(x) = \text{sign}\left(\sum_{t=1}^{T} \alpha_t h_t(x)\right) \tag{5}
$$

### Ejemplo Numérico

Dataset: 5 ejemplos, clasificación binaria {-1, +1}

| i | $x_i$ | $y_i$ | $w_i^{(1)}$ |
|---|-------|-------|-------------|
| 1 | 0.5   | +1    | 0.2         |
| 2 | 1.0   | +1    | 0.2         |
| 3 | 1.5   | -1    | 0.2         |
| 4 | 2.0   | -1    | 0.2         |
| 5 | 2.5   | -1    | 0.2         |

**Iteración 1:**
- Clasificador $h_1$: threshold en $x=1.25$ → predice +1 si $x < 1.25$
- Errores: ejemplo 3 mal clasificado
- $\epsilon_1 = 0.2$
- $\alpha_1 = 0.5 \ln(4) = 0.693$
- Actualizar pesos: aumentar $w_3$, reducir otros

---

## 3.2 Gradient Boosting

**Idea:** En cada iteración, entrenar un modelo para predecir los **residuos** (errores) del modelo actual.

### Marco General

Queremos minimizar:
$$
L = \sum_{i=1}^{n} \ell(y_i, F(x_i)) \tag{6}
$$

Donde $F(x)$ es nuestro modelo ensemble y $\ell$ es una función de pérdida.

### Algoritmo (caso de regresión)

**Inicialización:**
$$
F_0(x) = \arg\min_\gamma \sum_{i=1}^{n} \ell(y_i, \gamma) \tag{7}
$$

Para MSE: $F_0(x) = \bar{y}$ (media)

**Para m = 1 to M:**

1. Calcular pseudo-residuos:
$$
r_{im} = -\left[\frac{\partial \ell(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}} \tag{8}
$$

Para MSE: $r_{im} = y_i - F_{m-1}(x_i)$ (residuos simples)

2. Entrenar árbol $h_m$ para predecir $r_{im}$:
$$
h_m = \arg\min_h \sum_{i=1}^{n} (r_{im} - h(x_i))^2 \tag{9}
$$

3. Actualizar modelo:
$$
F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x) \tag{10}
$$

Donde $\nu$ es el **learning rate** (típicamente 0.01-0.3)

**Predicción final:**
$$
F_M(x) = F_0(x) + \nu \sum_{m=1}^{M} h_m(x) \tag{11}
$$

### ¿Por qué funciona?

**Analogía con Gradient Descent:**

En gradient descent:
$$
\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)
$$

En gradient boosting:
$$
F_{m}(x) = F_{m-1}(x) - \nu \cdot (\text{gradiente aproximado por } h_m)
$$

**¡Hacemos gradient descent en el espacio de funciones!**

---

## 3.3 XGBoost (Extreme Gradient Boosting)

**Mejoras sobre Gradient Boosting:**

### 1. Función Objetivo con Regularización

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} \ell(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t) \tag{12}
$$

Donde el término de regularización:
$$
\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2 \tag{13}
$$

- $T$: número de hojas
- $w_j$: peso de la hoja $j$
- $\gamma$: penalización por complejidad (número de hojas)
- $\lambda$: regularización L2 en pesos

### 2. Aproximación de Taylor de Segundo Orden

$$
\mathcal{L}^{(t)} \approx \sum_{i=1}^{n} [g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)] + \Omega(f_t) \tag{14}
$$

Donde:
- $g_i = \frac{\partial \ell}{\partial \hat{y}^{(t-1)}}$ (primer orden)
- $h_i = \frac{\partial^2 \ell}{\partial (\hat{y}^{(t-1)})^2}$ (segundo orden)

### 3. Peso Óptimo de Hojas

Para una hoja $j$:
$$
w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda} \tag{15}
$$

### 4. Ganancia de Split

$$
\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda}\right] - \gamma \tag{16}
$$

Donde $G_L = \sum_{i \in I_L} g_i$, $H_L = \sum_{i \in I_L} h_i$

**Interpretación:** Solo hacer split si la ganancia supera el costo de regularización $\gamma$

---

## 3.4 LightGBM

**Innovaciones para eficiencia:**

### 1. Gradient-based One-Side Sampling (GOSS)

- Mantener todas las muestras con gradientes grandes (|g| > threshold)
- Muestrear aleatoriamente muestras con gradientes pequeños
- Compensar con pesos al calcular ganancia

**Razón:** Muestras bien clasificadas (gradiente pequeño) contribuyen menos a ganancia de información

### 2. Exclusive Feature Bundling (EFB)

- Agrupar features mutuamente exclusivas (ej: one-hot encoded)
- Reducir dimensionalidad efectiva

### 3. Leaf-wise Growth

**Tradicional (level-wise):** Dividir todos los nodos de un nivel

**LightGBM (leaf-wise):** Dividir la hoja con mayor ganancia

```
Level-wise:           Leaf-wise:
       A                    A
      / \                  / \
     B   C                B   C
    / \  / \               \   \
   D E  F G                 D   E
                             \   \
                              F   G
```

**Ventaja:** Árboles más profundos con menos nodos → mayor accuracy, menos memoria

In [ ]:
# Demostración de cómo AdaBoost ajusta pesos
# Simulación simple

# Dataset simple
X_simple = np.array([[0.5], [1.0], [1.5], [2.0], [2.5]])
y_simple = np.array([1, 1, -1, -1, -1])

# Inicialización
n_samples = len(y_simple)
weights = np.ones(n_samples) / n_samples

print("🎯 Simulación de AdaBoost:\n")
print("Dataset inicial:")
for i in range(n_samples):
    print(f"  Ejemplo {i+1}: x={X_simple[i,0]:.1f}, y={y_simple[i]:+d}, peso={weights[i]:.3f}")

# Predicciones de un clasificador débil
predictions = np.array([1, 1, 1, -1, -1])  # Threshold en 1.75

# Calcular error
incorrect = (predictions != y_simple)
error = np.sum(weights[incorrect])

print(f"\nClasificador débil (threshold=1.75):")
print(f"  Predicciones: {predictions}")
print(f"  Incorrectas: índices {np.where(incorrect)[0]}")
print(f"  Error ponderado: {error:.3f}")

# Calcular alpha
alpha = 0.5 * np.log((1 - error) / error)
print(f"  Alpha (peso del clasificador): {alpha:.3f}")

# Actualizar pesos
new_weights = weights * np.exp(-alpha * y_simple * predictions)
new_weights /= new_weights.sum()  # Normalizar

print(f"\nPesos actualizados:")
for i in range(n_samples):
    arrow = "↑" if new_weights[i] > weights[i] else "↓"
    print(f"  Ejemplo {i+1}: {weights[i]:.3f} → {new_weights[i]:.3f} {arrow}")

print("\n💡 Los ejemplos mal clasificados aumentan su peso para la siguiente iteración")

---
## 💻 4. Implementación Desde Cero

In [ ]:
class AdaBoost:
    """
    Implementación simplificada de AdaBoost para clasificación binaria.
    
    Parameters:
    -----------
    n_estimators : int, default=50
        Número de clasificadores débiles
    learning_rate : float, default=1.0
        Factor de shrinkage para cada clasificador
    """
    
    def __init__(self, n_estimators=50, learning_rate=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        
        # Almacenar clasificadores y sus pesos
        self.estimators = []
        self.estimator_weights = []
        self.estimator_errors = []
    
    def fit(self, X, y):
        """
        Entrena el modelo AdaBoost.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        y : array-like, shape (n_samples,)
            Debe ser {-1, +1}
        """
        n_samples = X.shape[0]
        
        # Convertir y a {-1, +1} si es necesario
        y = np.where(y == 0, -1, y)
        
        # Inicializar pesos uniformemente
        sample_weights = np.ones(n_samples) / n_samples
        
        print(f"🚀 Entrenando AdaBoost con {self.n_estimators} clasificadores...\n")
        
        for iboost in range(self.n_estimators):
            # 1. Entrenar clasificador débil con pesos actuales
            # Usamos DecisionTreeClassifier con max_depth=1 (decision stump)
            estimator = DecisionTreeClassifier(
                max_depth=1,
                random_state=42 + iboost
            )
            
            estimator.fit(X, y, sample_weight=sample_weights)
            y_pred = estimator.predict(X)
            
            # 2. Calcular error ponderado
            incorrect = (y_pred != y)
            estimator_error = np.sum(sample_weights[incorrect]) / np.sum(sample_weights)
            
            # Si el clasificador es peor que random, parar
            if estimator_error >= 0.5:
                if len(self.estimators) == 0:
                    print(f"⚠️  Clasificador {iboost+1}: Error {estimator_error:.4f} >= 0.5, deteniendo...")
                break
            
            # 3. Calcular peso del estimador (alpha)
            estimator_weight = self.learning_rate * 0.5 * np.log(
                (1 - estimator_error) / (estimator_error + 1e-10)
            )
            
            # 4. Actualizar pesos de muestras
            sample_weights *= np.exp(-estimator_weight * y * y_pred)
            sample_weights /= np.sum(sample_weights)  # Normalizar
            
            # Guardar
            self.estimators.append(estimator)
            self.estimator_weights.append(estimator_weight)
            self.estimator_errors.append(estimator_error)
            
            # Logging
            if (iboost + 1) % 10 == 0:
                train_pred = self.predict(X)
                train_acc = np.mean(train_pred == y)
                print(f"  Iteración {iboost+1}: Error={estimator_error:.4f}, "
                      f"Alpha={estimator_weight:.4f}, Train Acc={train_acc:.4f}")
        
        print(f"\n✅ Entrenamiento completado con {len(self.estimators)} clasificadores")
        return self
    
    def predict(self, X):
        """
        Predice clases para X.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        y_pred : array, shape (n_samples,)
        """
        # Votación ponderada
        predictions = np.zeros(X.shape[0])
        
        for estimator, weight in zip(self.estimators, self.estimator_weights):
            predictions += weight * estimator.predict(X)
        
        return np.sign(predictions)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y = np.where(y == 0, -1, y)
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase AdaBoost definida")

In [ ]:
# Probar con Breast Cancer dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Convertir a {-1, +1}
y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar nuestro AdaBoost
ada_custom = AdaBoost(n_estimators=50, learning_rate=1.0)
ada_custom.fit(X_train, y_train)

In [ ]:
# Evaluar
train_acc = ada_custom.score(X_train, y_train)
test_acc = ada_custom.score(X_test, y_test)

print(f"📊 Resultados:")
print(f"   Train Accuracy: {train_acc:.4f}")
print(f"   Test Accuracy: {test_acc:.4f}")

# Visualizar evolución del error
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, len(ada_custom.estimator_errors) + 1)),
    y=ada_custom.estimator_errors,
    mode='lines+markers',
    name='Error del Estimador',
    line=dict(color='red', width=2)
))

fig.update_layout(
    title="Error de Clasificadores Débiles en cada Iteración",
    xaxis_title="Iteración",
    yaxis_title="Error Ponderado",
    template="plotly_white",
    font=dict(size=12)
)

fig.show()

print("\n💡 Los errores disminuyen porque cada nuevo clasificador se enfoca en casos difíciles")

---
## 🏭 5. Versión con Frameworks (Scikit-learn, XGBoost, LightGBM)

In [ ]:
# Preparar datos (volver a {0, 1})
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Scikit-learn AdaBoost
ada_sklearn = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42)
ada_sklearn.fit(X_train, y_train)

# 2. Scikit-learn Gradient Boosting
gb_sklearn = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, 
                                       max_depth=3, random_state=42)
gb_sklearn.fit(X_train, y_train)

# 3. XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

# 4. LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
lgb_model.fit(X_train, y_train)

# Comparar resultados
models = {
    'AdaBoost (sklearn)': ada_sklearn,
    'Gradient Boosting (sklearn)': gb_sklearn,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model
}

print("📊 Comparación de Algoritmos de Boosting\n")
print("="*80)
print(f"{'Modelo':<30} {'Train Acc':<15} {'Test Acc':<15} {'ROC-AUC':<15}")
print("="*80)

for name, model in models.items():
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    # ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = 0
    
    print(f"{name:<30} {train_acc:<15.4f} {test_acc:<15.4f} {roc_auc:<15.4f}")

print("="*80)

In [ ]:
# Visualizar Feature Importances de XGBoost
importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1][:10]  # Top 10

fig = go.Figure()

fig.add_trace(go.Bar(
    x=importances[indices],
    y=[cancer.feature_names[i] for i in indices],
    orientation='h',
    marker=dict(color=importances[indices], colorscale='Viridis')
))

fig.update_layout(
    title="Top 10 Feature Importances - XGBoost",
    xaxis_title="Importance",
    yaxis_title="Feature",
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

In [ ]:
# Tunear hiperparámetros de XGBoost con GridSearchCV
print("🔧 Tuneando hiperparámetros de XGBoost...\n")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0]
}

xgb_grid = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

grid_search = GridSearchCV(
    xgb_grid,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Mejores hiperparámetros:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Mejor CV Score: {grid_search.best_score_:.4f}")
print(f"   Test Accuracy: {grid_search.score(X_test, y_test):.4f}")

### Comparación: Boosting vs Bagging

| Aspecto | Random Forest (Bagging) | Boosting |
|---------|------------------------|----------|
| **Entrenamiento** | Paralelo | Secuencial |
| **Dependencia** | Árboles independientes | Cada árbol depende del anterior |
| **Objetivo** | Reducir varianza | Reducir bias y varianza |
| **Overfitting** | Menos propenso | Más propenso si no se regulariza |
| **Velocidad** | Más rápido (paralelizable) | Más lento (secuencial) |
| **Robustez** | Robusto a outliers | Sensible a outliers |
| **Tuning** | Poco sensible a hiperparámetros | Requiere cuidadoso tuning |
| **Performance** | Bueno out-of-the-box | Excelente con tuning |

### Cuándo usar cada uno

**Usa Random Forests si:**
- Quieres un modelo robusto sin mucho tuning
- Los datos tienen muchos outliers o ruido
- Necesitas paralelizar el entrenamiento
- Interpretabilidad es importante (feature importances más estables)

**Usa Boosting (XGBoost/LightGBM) si:**
- Quieres el máximo performance posible
- Puedes dedicar tiempo a tunear hiperparámetros
- Los datos están limpios (o puedes manejar outliers)
- Participas en competencias de ML (Kaggle, etc.)

---
## 🎯 6. Ejercicios

### 🟢 Ejercicio 1: Efecto del Learning Rate

Experimenta con diferentes learning rates en XGBoost y observa su efecto.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Entender el rol del learning rate
    
    Instrucciones:
    1. Usa Breast Cancer dataset (ya cargado)
    2. Entrena XGBoost con learning_rate = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
    3. Usa n_estimators=100, max_depth=3
    4. Calcula test accuracy para cada learning rate
    5. Visualiza la relación learning_rate vs accuracy
    
    Returns:
    --------
    results : dict
        {learning_rate: test_accuracy}
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# results = ejercicio_1()
# print("\n💡 Learning rates muy altos pueden causar overfitting")
# print("   Learning rates muy bajos pueden necesitar más árboles")

### 🟡 Ejercicio 2: Early Stopping

Implementa early stopping para prevenir overfitting en XGBoost.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Usar early stopping para optimizar entrenamiento
    
    Instrucciones:
    1. Divide los datos en train/validation/test (60/20/20)
    2. Entrena XGBoost con n_estimators=500
    3. Usa early_stopping_rounds=10 (detener si no mejora en 10 rondas)
    4. Evalúa en test set
    5. Compara con modelo sin early stopping
    
    Returns:
    --------
    results : dict
        {
            'best_iteration': int,
            'test_acc_with_early_stop': float,
            'test_acc_without_early_stop': float
        }
    """
    # TODO: Tu código aquí
    # Pista: usa eval_set y early_stopping_rounds en fit()
    
    pass

# Descomentar para probar
# results = ejercicio_2()
# print(f"\nMejor iteración: {results['best_iteration']}")
# print(f"Test Acc (con early stop): {results['test_acc_with_early_stop']:.4f}")
# print(f"Test Acc (sin early stop): {results['test_acc_without_early_stop']:.4f}")

### 🔴 Ejercicio 3: Comparación Completa con Tuning

Compara Random Forest, XGBoost y LightGBM con tuning completo de hiperparámetros.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Realizar una comparación rigurosa de algoritmos ensemble
    
    Instrucciones:
    1. Usa Breast Cancer dataset
    2. Para cada modelo (RandomForest, XGBoost, LightGBM):
       a) Define un grid de hiperparámetros razonable
       b) Usa GridSearchCV con 5-fold CV
       c) Evalúa en test set
    3. Compara:
       - Test accuracy
       - ROC-AUC
       - Tiempo de entrenamiento
       - Número de parámetros óptimos
    4. Crea visualizaciones comparativas
    
    Returns:
    --------
    results : dict
        Diccionario con métricas de cada modelo
    """
    # TODO: Tu código aquí
    # Pista: usa time.time() para medir tiempo de entrenamiento
    
    pass

# Descomentar para probar
# results = ejercicio_3()
# print("\n🏆 Comparación Final:")
# for model_name, metrics in results.items():
#     print(f"\n{model_name}:")
#     for metric, value in metrics.items():
#         print(f"  {metric}: {value}")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **Boosting entrena modelos secuencialmente**
   - Cada modelo se enfoca en corregir errores del anterior
   - Reduce tanto bias como varianza
   - Votación ponderada (mejores modelos pesan más)

2. **AdaBoost**
   - Ajusta pesos de muestras según errores
   - Clasificadores débiles → clasificador fuerte
   - Simple pero efectivo

3. **Gradient Boosting**
   - Marco general: gradient descent en espacio de funciones
   - Cada árbol predice residuos (gradientes negativos)
   - Muy flexible (cualquier loss function diferenciable)

4. **XGBoost**
   - Regularización explícita (L1/L2)
   - Aproximación de segundo orden (Newton method)
   - Highly optimizado (paralelización, cache-awareness)
   - Maneja missing values automáticamente

5. **LightGBM**
   - GOSS: muestreo inteligente basado en gradientes
   - EFB: bundling de features
   - Leaf-wise growth: más profundo con menos nodos
   - Extremadamente rápido en datasets grandes

6. **Hiperparámetros críticos:**
   - `learning_rate`: controla cuánto contribuye cada árbol (0.01-0.3)
   - `n_estimators`: número de árboles (50-500+)
   - `max_depth`: profundidad de árboles (3-10)
   - `subsample`: fracción de datos para cada árbol (0.5-1.0)
   - `colsample_bytree`: fracción de features (0.5-1.0)
   - Regularización: `lambda` (L2), `alpha` (L1), `gamma` (complejidad)

7. **Trade-offs:**
   - ✅ Performance excepcional con tuning
   - ✅ Maneja features heterogéneas bien
   - ✅ Feature importances built-in
   - ❌ Propenso a overfitting sin regularización
   - ❌ Sensible a outliers
   - ❌ Requiere más tuning que Random Forests
   - ❌ Menos interpretable

---

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"A Decision-Theoretic Generalization of On-Line Learning"** - Freund & Schapire (1997)
  - Paper original de AdaBoost
  - Premio Gödel 2003
  - https://www.sciencedirect.com/science/article/pii/S002200009791504X

- **"Greedy Function Approximation: A Gradient Boosting Machine"** - Friedman (2001)
  - Marco teórico de Gradient Boosting
  - Fundamental para entender XGBoost/LightGBM

- **"XGBoost: A Scalable Tree Boosting System"** - Chen & Guestrin (2016)
  - Paper de XGBoost (KDD 2016)
  - https://arxiv.org/abs/1603.02754

- **"LightGBM: A Highly Efficient Gradient Boosting Decision Tree"** - Ke et al. (2017)
  - Paper de LightGBM (NIPS 2017)
  - https://papers.nips.cc/paper/6907-lightgbm-a-highly-efficient-gradient-boosting-decision-tree

#### 📖 Libros Recomendados

- **"The Elements of Statistical Learning"** - Hastie, Tibshirani, Friedman
  - Capítulo 10: Boosting and Additive Trees
  - Tratamiento matemático riguroso

- **"Ensemble Methods"** - Zhi-Hua Zhou (2012)
  - Libro completo sobre ensemble learning
  - Cubre teoría y práctica

#### 🎥 Videos Recomendados

- **StatQuest: AdaBoost and Gradient Boost** - Josh Starmer
  - Explicaciones visuales excelentes
  - https://www.youtube.com/watch?v=LsK-xG1cLYA

- **XGBoost Tutorial** - Tianqi Chen (creador de XGBoost)
  - Presentación del autor original

#### 💻 Documentación Oficial

- [XGBoost Documentation](https://xgboost.readthedocs.io/)
- [LightGBM Documentation](https://lightgbm.readthedocs.io/)
- [Scikit-learn: Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble.html)

#### 🏆 Recursos de Kaggle

- [Complete Guide to XGBoost](https://www.kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning)
- [LightGBM vs XGBoost](https://www.kaggle.com/code/prasunmishra/lightgbm-vs-xgboost-who-will-win-the-race)

---

### 🤔 Preguntas para Reflexionar

1. **¿Por qué boosting es más propenso a overfitting que bagging?**
   - Pista: Piensa en cómo se enfocan en ejemplos difíciles

2. **¿Cuándo usarías learning rate bajo vs alto?**
   - Considera la relación con n_estimators

3. **¿Por qué XGBoost usa aproximación de segundo orden?**
   - Relaciona con Newton method vs gradient descent

4. **¿En qué casos LightGBM es mejor que XGBoost?**
   - Piensa en tamaño de datos, dimensionalidad, tiempo

---

## ➡️ Próximo Paso

En el siguiente notebook, **08. Support Vector Machines (SVM)**, aprenderemos sobre:

- **Márgenes máximos**: Encontrar el mejor hiperplano separador
- **Kernel trick**: Transformar datos a espacios de mayor dimensión
- **Support vectors**: Solo las muestras críticas importan
- **Parámetro C**: Trade-off entre margen y errores
- **Datos no linealmente separables**: RBF, polynomial kernels

**Cambio de paradigma:** Pasamos de ensemble methods a un enfoque geométrico de clasificación basado en márgenes.

---

<div align="center">

**🚀 Del ensemble secuencial al margen máximo 🚀**

**Continúa con: [08. Support Vector Machines](08-svm.ipynb)**

[← 06. Random Forests](06-random-forests.ipynb) | [08. SVM →](08-svm.ipynb)

</div>